## Virtual Environment

Create and activate a virtual environment:

```bash
python -m venv irony
source venv/bin/activate  # On Windows: venv\Scripts\activate
```

or using conda
```
conda create -n irony python=3.14.3
conda activate irony
```
Install kernel

```bash
pip install --user ipykernel
python -m ipykernel install --user --name=irony
```

## Install Dependencies

```bash
pip install -r requirements.txt
```

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"
)

def check_model_loaded(model, tokenizer):
    test_input = "Hello, my name is"
    
    inputs = tokenizer(test_input, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10)
    
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Input:  {test_input}")
    print(f"Output: {decoded}")
    print(f"Device: {model.device}")
    print(f"Dtype:  {model.dtype}")

check_model_loaded(model, tokenizer)


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


Input:  Hello, my name is
Output: Hello, my name is Anya Petrova, and I’m a software
Device: mps:0
Dtype:  torch.bfloat16


In [7]:
def run_zero_shot(prompt: str, max_new_tokens: int = 258) -> str:
    # OLMo2 Instruct uses a chat template
    print(prompt)
    messages = [{"role": "user", "content": prompt}]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    print(input_ids)
        # If it's a BatchEncoding, unpack it; if it's already a tensor, keep it
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids["input_ids"]
    
    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    response = tokenizer.decode(
        output[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    print(response)
    return response.strip()

In [ ]:
import pandas as pd
from tqdm import tqdm

# Load and process all 3 files
files = {
    # "condition1": "./data/Condition1(context_richness).csv",
    # "condition2": "./data/Condition2(common_ground).csv",
    "condition3": "./data/Condition3(prompting_style).csv",
}

for name, path in files.items():
    print(f"\nProcessing {name}...")
    
    df = pd.read_csv(path)
    print(df.head())  # sanity check
    
    tqdm.pandas()
    df["model_output"] = df["full_prompt"].progress_apply(run_zero_shot)
    
    output_path = f"results_gemma_zeroshot_{name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

print("\nAll done!")


Processing condition3...
      item_id prompt_level irony_label                 base_utterance  \
0   C3_01_L_I          low      ironic  That went exactly as planned.   
1  C3_01_L_NI          low  non-ironic  That went exactly as planned.   
2   C3_01_H_I         high      ironic  That went exactly as planned.   
3  C3_01_H_NI         high  non-ironic  That went exactly as planned.   
4   C3_02_L_I          low      ironic    I love how this turned out.   

                                         full_prompt  
0  Consider the following before answering:\n (1)...  
1  Consider the following before answering:\n (1)...  
2  Consider the following before answering:\n (1)...  
3  Consider the following before answering:\n (1)...  
4  Consider the following before answering:\n (1)...  


  0%|                                                   | 0/100 [00:00<?, ?it/s]

Consider the following before answering:
 (1) What do the individual words in this statement literally mean?
 (2) Taking the words at face value, does the statement describe something positive or negative?
 (3) Based purely on the meaning of the words, what is the most straightforward interpretation?

Is the following statement ironic? Answer with yes or no.
 "That went exactly as planned."
{'input_ids': tensor([[     2,    105,   2364,    107,  24501,    506,   2269,   1680,  38020,
         236787,    107,    568, 236770, 236768,   2900,    776,    506,   3141,
           4171,    528,    672,   5456,  17720,   2689, 236881,    107,    568,
         236778, 236768,  32110,    506,   4171,    657,   3392,   1550, 236764,
           1677,    506,   5456,   9779,   2613,   4414,    653,   5676, 236881,
            107,    568, 236800, 236768,  18186,  27862,    580,    506,   6590,
            529,    506,   4171, 236764,   1144,    563,    506,   1346,  24267,
          16823, 236881, 

In [ ]:
torch.cuda.is_available()
print("Done! Results saved.")